# Module 4: Classical Time Series Models (ETS, SARIMAX)

This notebook fits **classical** forecasting models on a *small SKU sample* and compares them to baselines.

Why a sample?
- ETS/SARIMAX are typically fit per SKU and can be slow/fragile at large scale.

## What you'll do
- Load feature-ready dataset from Module 2
- Create a time-based split (same idea as Module 3)
- Fit ETS and SARIMAX per SKU (sample)
- Evaluate with MAE/RMSE/WAPE/sMAPE
- Run residual diagnostics (ACF + Ljung–Box)
- Save a comparison report


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../')

from src.models.baseline import naive_forecast, moving_average_forecast
from src.models.classical import ets_holt_winters_forecast, sarimax_forecast, sarimax_fit_residuals
from src.evaluation.metrics import mae, rmse, wape, smape

import matplotlib.pyplot as plt

print('Imports OK')


## Load data

We use the feature-ready dataset created in Module 2:
- `data/processed/featured_sales_data.csv`


In [ ]:
data_path_candidates = [
    Path('../data/processed/featured_sales_data.csv'),
    Path('data/processed/featured_sales_data.csv'),
]

data_path = next((p for p in data_path_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find featured_sales_data.csv. Run Module 2 to generate it: notebooks/02_data_cleaning.ipynb"
    )

df = pd.read_csv(data_path, parse_dates=['date'])
print(f"Loaded: {data_path}")
print(df.shape)
df.head()


## Define split and SKU sample

We’ll forecast the next `HORIZON` days after a cutoff date.

Because classical models are slower, we’ll fit them on a sample of SKUs:
- 5 top-volume SKUs
- 5 random SKUs


In [ ]:
SKU_COL = 'sku_id'
DATE_COL = 'date'
TARGET_COL = 'units_sold'

HORIZON = 14
max_date = df[DATE_COL].max()
cutoff = max_date - pd.Timedelta(days=HORIZON)

train = df[df[DATE_COL] <= cutoff].copy()
test = df[(df[DATE_COL] > cutoff) & (df[DATE_COL] <= cutoff + pd.Timedelta(days=HORIZON))].copy()

# SKU sample
sku_volume = train.groupby(SKU_COL)[TARGET_COL].sum().sort_values(ascending=False)
top_skus = sku_volume.head(5).index.tolist()
all_skus = train[SKU_COL].unique()
np.random.seed(42)
rand_skus = np.random.choice([s for s in all_skus if s not in top_skus], size=5, replace=False).tolist()

sku_sample = top_skus + rand_skus
print('Cutoff:', cutoff.date(), 'Horizon:', HORIZON)
print('SKU sample:', sku_sample)


## Fit models per SKU and score

We’ll compare:
- Baselines: naive, moving average (7)
- Classical: ETS (weekly seasonality), SARIMAX (weekly seasonality)

We’ll compute metrics per SKU and overall.


In [ ]:
def score_one_sku(sku: str):
    g_train = train[train[SKU_COL] == sku].sort_values(DATE_COL)
    g_test = test[test[SKU_COL] == sku].sort_values(DATE_COL)

    y_train = g_train[TARGET_COL].to_numpy(dtype=float)
    y_true = g_test[TARGET_COL].to_numpy(dtype=float)

    # Baselines
    y_pred_naive = naive_forecast(y_train, horizon=HORIZON).y_pred
    y_pred_ma7 = moving_average_forecast(y_train, horizon=HORIZON, window=7).y_pred

    # Classical
    try:
        y_pred_ets = ets_holt_winters_forecast(y_train, horizon=HORIZON, seasonal_periods=7).y_pred
    except Exception:
        y_pred_ets = np.full(HORIZON, np.nan)

    try:
        y_pred_sarimax = sarimax_forecast(y_train, horizon=HORIZON, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7)).y_pred
    except Exception:
        y_pred_sarimax = np.full(HORIZON, np.nan)

    def _metrics(y_pred):
        if np.any(np.isnan(y_pred)):
            return {'MAE': np.nan, 'RMSE': np.nan, 'WAPE': np.nan, 'sMAPE': np.nan}
        return {
            'MAE': mae(y_true, y_pred),
            'RMSE': rmse(y_true, y_pred),
            'WAPE': wape(y_true, y_pred),
            'sMAPE': smape(y_true, y_pred),
        }

    rows = []
    for method, y_pred in [
        ('naive', y_pred_naive),
        ('moving_average_7', y_pred_ma7),
        ('ets_hw_7', y_pred_ets),
        ('sarimax_111_111_7', y_pred_sarimax),
    ]:
        m = _metrics(y_pred)
        m.update({'sku_id': sku, 'method': method})
        rows.append(m)

    return pd.DataFrame(rows)

per_sku_scores = pd.concat([score_one_sku(s) for s in sku_sample], ignore_index=True)
per_sku_scores


In [ ]:
# Overall comparison (average across sampled SKUs)
overall = per_sku_scores.groupby('method')[['MAE','RMSE','WAPE','sMAPE']].mean().sort_values('WAPE')
overall


## Residual diagnostics (SARIMAX)

We’ll fit SARIMAX on one SKU and inspect residuals:
- residual series
- ACF plot
- Ljung–Box test

If residuals show strong autocorrelation, the model is missing structure.


In [ ]:
diag_sku = sku_sample[0]
y_train = train[train[SKU_COL] == diag_sku].sort_values(DATE_COL)[TARGET_COL].to_numpy(dtype=float)

fit, resid = sarimax_fit_residuals(y_train, order=(1,1,1), seasonal_order=(1,1,1,7))
print('Diagnostics SKU:', diag_sku)
print('Residuals mean:', np.mean(resid).round(4), 'std:', np.std(resid).round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(resid, linewidth=0.8)
axes[0].set_title('SARIMAX Residuals (time order)')
axes[0].set_xlabel('t')
axes[0].set_ylabel('residual')

# ACF
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(resid, ax=axes[1], lags=40)
axes[1].set_title('Residual ACF (lags)')

plt.tight_layout()
plt.show()

# Ljung-Box
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(resid, lags=[7, 14, 21, 28], return_df=True)
lb


In [ ]:
# Save reports
out_dir = Path('../outputs/reports')
out_dir.mkdir(parents=True, exist_ok=True)

per_sku_out = out_dir / 'module4_classical_per_sku_metrics.csv'
per_sku_scores.to_csv(per_sku_out, index=False)
print('Saved:', per_sku_out)

overall_out = out_dir / 'module4_classical_summary.csv'
overall.reset_index().to_csv(overall_out, index=False)
print('Saved:', overall_out)
